In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [9]:
df = pd.read_csv('steam.csv')

In [10]:
df.isna().sum()

appid                0
name                 0
release_date         0
english              0
developer            1
publisher           14
platforms            0
required_age         0
categories           0
genres               0
steamspy_tags        0
achievements         0
positive_ratings     0
negative_ratings     0
average_playtime     0
median_playtime      0
owners               0
price                0
dtype: int64

In [11]:
df.dropna(inplace=True)
df.isna().sum()

appid               0
name                0
release_date        0
english             0
developer           0
publisher           0
platforms           0
required_age        0
categories          0
genres              0
steamspy_tags       0
achievements        0
positive_ratings    0
negative_ratings    0
average_playtime    0
median_playtime     0
owners              0
price               0
dtype: int64

In [13]:
df.reset_index(drop=True)

,appid,name,release_date,english,developer,publisher,platforms,required_age,categories,genres,steamspy_tags,achievements,positive_ratings,negative_ratings,average_playtime,median_playtime,owners,price
0,10,Counter-Strike,2000-11-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Online Multi-Player;Local Multi-P...,Action,Action;FPS;Multiplayer,0,124534,3339,17612,317,10000000-20000000,7.19
1,20,Team Fortress Classic,1999-04-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Online Multi-Player;Local Multi-P...,Action,Action;FPS;Multiplayer,0,3318,633,277,62,5000000-10000000,3.99
2,30,Day of Defeat,2003-05-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Valve Anti-Cheat enabled,Action,FPS;World War II;Multiplayer,0,3416,398,187,34,5000000-10000000,3.99
3,40,Deathmatch Classic,2001-06-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Online Multi-Player;Local Multi-P...,Action,Action;FPS;Multiplayer,0,1273,267,258,184,5000000-10000000,3.99
4,50,Half-Life: Opposing Force,1999-11-01,1,Gearbox Software,Valve,windows;mac;linux,0,Single-player;Multi-player;Valve Anti-Cheat en...,Action,FPS;Action;Sci-fi,0,5250,288,624,415,5000000-10000000,3.99
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27056,1065230,Room of Pandora,2019-04-24,1,SHEN JIAWEI,SHEN JIAWEI,windows,0,Single-player;Steam Achievements,Adventure;Casual;Indie,Adventure;Indie;Casual,7,3,0,0,0,0-20000,2.09
27057,1065570,Cyber Gun,2019-04-23,1,Semyon Maximov,BekkerDev Studio,windows,0,Single-player,Action;Adventure;Indie,Action;Indie;Adventure,0,8,1,0,0,0-20000,1.69
27058,1065650,Super Star Blast,2019-04-24,1,EntwicklerX,EntwicklerX,windows,0,Single-player;Multi-player;Co-op;Shared/Split ...,Action;Casual;Indie,Action;Indie;Casual,24,0,1,0,0,0-20000,3.99
27059,1066700,New Yankee 7: Deer Hunters,2019-04-17,1,Yustas Game Studio,Alawar Entertainment,windows;mac,0,Single-player;Steam Cloud,Adventure;Casual;Indie,Indie;Casual;Adventure,0,2,0,0,0,0-20000,5.19


In [17]:
def combine_features(row):
    return str(row['developer']) + " " + str(row['categories']) + " " + str(row['genres']) + " " + str(row['steamspy_tags'])

df['combined_features'] = df.apply(combine_features, axis=1)
df['combined_features'] = df['combined_features'].str.replace(';', ' ')

In [19]:
cv = TfidfVectorizer(stop_words='english')
feature_vectors = cv.fit_transform(df['combined_features'])
similarity_matrix = cosine_similarity(feature_vectors)

In [30]:
def get_recommendations(game_name, max_price=None, required_platform=None, num_recommendations=5):
    try:
        game_index = df[df['name'].str.contains(game_name, case=False, na=False)].index[0]
    except IndexError:
        return
    similarity_scores = list(enumerate(similarity_matrix[game_index]))
    
    sorted_similar_games = sorted(similarity_scores, key=lambda x: x[1], reverse=True)[1:]

    recommended_games = []
    
    for game in sorted_similar_games:
        index = game[0]
        title = df.iloc[index]['name']
        price = df.iloc[index]['price']
        platforms = df.iloc[index]['platforms']
        
        if max_price is not None and price > max_price:
            continue
            
        if required_platform is not None and required_platform not in platforms:
            continue
            
        recommended_games.append(f"{title} (${price})")
        
        if len(recommended_games) >= num_recommendations:
            break
            
    return recommended_games

In [31]:
print("Recommendations for Counter-Strike (Under $10, Mac compatible):")
print(get_recommendations('Counter-Strike', max_price=10.00, required_platform='mac'))

Recommendations for Counter-Strike (Under $10, Mac compatible):
['Team Fortress Classic ($3.99)', 'Deathmatch Classic ($3.99)', 'Ricochet ($3.99)', 'Counter-Strike: Condition Zero ($7.19)', 'Half-Life Deathmatch: Source ($0.0)']
